In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

#load .env
load_dotenv()

with open("config/config.json","r") as f:
    config = json.load(f)

print(config)

{'llm': {'openai_model': 'gpt-4o-mini', 'gemini_model': 'gemini-1.5-flash', 'temperature': 0.3}, 'evaluation': {'criteria': ['tone_empathy', 'knowledge_accuracy', 'resolution_quality'], 'score_range': [1, 5]}, 'classification': {'labels': ['billing', 'claims', 'complaint', 'general_query']}}


In [29]:
df = pd.read_csv("data/transcripts.csv")

print("Data Loaded")
print(df.head())

Data Loaded
   call_id agent_name                                         transcript  \
0        1      Alice  Customer: I was charged twice for my premium t...   
1        2        Bob  Customer: My claim has been pending for 2 week...   
2        3    Charlie  Customer: This is the third time I'm calling! ...   
3        4      Diana  Customer: Can you explain my coverage details?...   

  expected_call_type  
0            billing  
1             claims  
2          complaint  
3      general_query  


In [30]:
df.tail(2)

,call_id,agent_name,transcript,expected_call_type
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint
3,4,Diana,Customer: Can you explain my coverage details?...,general_query


In [31]:
df.head()

,call_id,agent_name,transcript,expected_call_type
0,1,Alice,Customer: I was charged twice for my premium t...,billing
1,2,Bob,Customer: My claim has been pending for 2 week...,claims
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint
3,4,Diana,Customer: Can you explain my coverage details?...,general_query


In [32]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

def load_llm(llm):
    provider = os.getenv("LLM_PROVIDER")

    if provider == "openai":
        llm = ChatOpenAI(model = config["llm"].get("openai_model", "gpt-4o-mini"), temperature = config["llm"].get("temperature", 0.3) )
    elif provider == "gemini":
        llm = ChatGoogleGenerativeAI(model = config["llm"].get("gemini_model", "gemini-1.5-flash"), temperature = config["llm"].get("temperature", 0.3) )
    else:
        raise ValueError("LLM_PROVIDER")
    return llm

llm = load_llm(config)

print(f"llm initialized using provider: {os.getenv('LLM_PROVIDER')}")

llm initialized using provider: openai


In [33]:
response = llm.invoke("Say 'setup successful' in one short sentence")

print(response)

content='Setup successful!' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 3, 'prompt_tokens': 16, 'total_tokens': 19, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_50c083517f', 'id': 'chatcmpl-EXqbOeQ0XZcLkLh6uVgD2A9X3wiTc', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a12bc5-9e6e-76c0-8c4e-bc1d54bbc3e9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 16, 'output_tokens': 3, 'total_tokens': 19, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [34]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


class ClassificationOutput(BaseModel):
    call_type: str = Field(description = "Type of customer call")
    confidence: float = Field(description = "Confidence score between 0 and 1")

parser = PydanticOutputParser(pydantic_object = ClassificationOutput)


#prompt template
prompt = PromptTemplate(
    template = """
    You are called classification assistant. 

    classify the following customer support transcript into one of the categories:
    {labels}

    Transcript:
    {transcript}

    {format_instructions}   
    """, 
    input_variables = ["transcript"], 
    partial_variables = {
        "format_instructions": parser.get_format_instructions(), 
            "labels": config["classification"]["labels"] 
                  }
)

#chain

classification_chain = prompt | llm | parser

#test chain with one transcript

# df.iloc[0]["transcript"]

response = classification_chain.invoke(df.iloc[0]["transcript"])

print(f"Classification result: {response}")

Classification result: call_type='billing' confidence=0.95


In [35]:
from tqdm import tqdm

results = []

for i , row in tqdm(df.iterrows(), total = len(df), desc = "Classifying Calls"):
    classification_chain = prompt | llm | parser

    try:
        response = classification_chain.invoke(row["transcript"])
        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": response.call_type,
            "confidence": response.confidence
        })

    except Exception as e:
        print(f"Error classifying call {row['call_id']}: {e}")
        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": None,
            "confidence": None
        })

results_df = pd.DataFrame(results)
df = df.merge(results_df, on = "call_id")

print(df)

Classifying Calls: 100%|██████████| 4/4 [00:03<00:00,  1.21it/s]

   call_id agent_name                                         transcript  \
0        1      Alice  Customer: I was charged twice for my premium t...   
1        2        Bob  Customer: My claim has been pending for 2 week...   
2        3    Charlie  Customer: This is the third time I'm calling! ...   
3        4      Diana  Customer: Can you explain my coverage details?...   

  expected_call_type predicted_call_type  confidence  
0            billing             billing        0.95  
1             claims              claims        0.95  
2          complaint           complaint        0.85  
3      general_query       general_query        0.90  


In [36]:
df

,call_id,agent_name,transcript,expected_call_type,predicted_call_type,confidence
0,1,Alice,Customer: I was charged twice for my premium t...,billing,billing,0.95
1,2,Bob,Customer: My claim has been pending for 2 week...,claims,claims,0.95
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint,complaint,0.85
3,4,Diana,Customer: Can you explain my coverage details?...,general_query,general_query,0.90


In [37]:
def route_call(call_type):
    if call_type == "billing":
        return ["knowledge_accuracy", "resolution_quality"]
    elif call_type == "claims":
        return ["knowledge_accuracy", "resolution_quality"]
    elif call_type == "complaint":
        return ["tone_empathy", "resolution_quality"]
    elif call_type == "general_query":
        return ["knowledge_accuracy"] 
    else:
        return ["knowledge_accuracy"] 

df["evaluation_plan"] = df["predicted_call_type"].apply(route_call)

print("Routing Applied\n")
print(df[["call_id", "predicted_call_type", "evaluation_plan"]])

Routing Applied

   call_id predicted_call_type                           evaluation_plan
0        1             billing  [knowledge_accuracy, resolution_quality]
1        2              claims  [knowledge_accuracy, resolution_quality]
2        3           complaint        [tone_empathy, resolution_quality]
3        4       general_query                      [knowledge_accuracy]


In [38]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

min_score, max_score = config["evaluation"]["score_range"]


class ToneEvaluationOutput(BaseModel):
    tone_empathy_score: int = Field(description = f"Tone and empathy score between {min_score} and {max_score}", ge = min_score, le = max_score)
    justification: str = Field(description = "Short reason for the score")

tone_parser = PydanticOutputParser(pydantic_object = ToneEvaluationOutput)


#prompt template
tone_prompt = PromptTemplate(
    template = """
    You are a quality assurance assistant for a customer support team.

    Evaluate the agent's tone and empathy in the following customer support transcript.
    Check whether the agent was polite, acknowledged the customer's feelings, and stayed calm and respectful.

    Score it from {min_score} (poor) to {max_score} (excellent).

    Transcript:
    {transcript}

    {format_instructions}
    """,
    input_variables = ["transcript"],
    partial_variables = {
        "format_instructions": tone_parser.get_format_instructions(),
            "min_score": min_score,
            "max_score": max_score
                  }
)

#chain

tone_chain = tone_prompt | llm | tone_parser

#test chain with one transcript

tone_response = tone_chain.invoke(df.iloc[0]["transcript"])

print(f"Tone evaluation result: {tone_response}")


Tone evaluation result: tone_empathy_score=3 justification="The agent acknowledged the customer's issue by offering to check it, but did not explicitly express empathy or understanding of the customer's frustration."


In [39]:
from tqdm import tqdm

tone_results = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating Tone"):
    # only evaluate calls whose evaluation plan includes tone_empathy
    if "tone_empathy" not in row["evaluation_plan"]:
        continue

    try:
        response = tone_chain.invoke(row["transcript"])
        tone_results.append({
            "call_id": row["call_id"],
            "tone_empathy_score": response.tone_empathy_score,
            "tone_justification": response.justification,
        })

    except Exception as e:
        print(f"Error evaluating tone for call {row['call_id']}: {e}")
        tone_results.append({
            "call_id": row["call_id"],
            "tone_empathy_score": None,
            "tone_justification": None,
        })

tone_df = pd.DataFrame(tone_results, columns=["call_id", "tone_empathy_score", "tone_justification"])
df = df.merge(tone_df, on="call_id", how="left")

print("Tone Evaluation Applied\n")
print(df[["call_id", "predicted_call_type", "tone_empathy_score", "tone_justification"]])


Evaluating Tone: 100%|██████████| 4/4 [00:01<00:00,  2.95it/s]

Tone Evaluation Applied

   call_id predicted_call_type  tone_empathy_score  \
0        1             billing                 NaN   
1        2              claims                 NaN   
2        3           complaint                 3.0   
3        4       general_query                 NaN   

                                  tone_justification  
0                                                NaN  
1                                                NaN  
2  The agent apologized for the inconvenience, wh...  
3                                                NaN  


In [40]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

min_score, max_score = config["evaluation"]["score_range"]


class ResolutionEvaluationOutput(BaseModel):
    resolution_quality_score: int = Field(description = f"Resolution quality score between {min_score} and {max_score}", ge = min_score, le = max_score)
    issue_resolved: bool = Field(description = "True if the customer's issue was fully resolved in the call, otherwise False")
    justification: str = Field(description = "Short reason for the score")

resolution_parser = PydanticOutputParser(pydantic_object = ResolutionEvaluationOutput)


#prompt template
resolution_prompt = PromptTemplate(
    template = """
    You are a quality assurance assistant for a customer support team.

    Evaluate how well the agent resolved the customer's issue in the following customer support transcript.
    Check whether the agent understood the problem, gave a clear solution or next steps, and whether the issue was actually resolved.

    Score it from {min_score} (poor) to {max_score} (excellent).

    Transcript:
    {transcript}

    {format_instructions}
    """,
    input_variables = ["transcript"],
    partial_variables = {
        "format_instructions": resolution_parser.get_format_instructions(),
            "min_score": min_score,
            "max_score": max_score
                  }
)

#chain

resolution_chain = resolution_prompt | llm | resolution_parser

#test chain with one transcript

resolution_response = resolution_chain.invoke(df.iloc[0]["transcript"])

print(f"Resolution evaluation result: {resolution_response}")


Resolution evaluation result: resolution_quality_score=2 issue_resolved=False justification="The agent did not provide a solution or next steps for the customer's issue regarding being charged twice; they only indicated they would check the issue."


In [41]:
from tqdm import tqdm

resolution_results = []

for i, row in tqdm(df.iterrows(), total = len(df), desc = "Evaluating Resolution"):
    # only evaluate calls whose evaluation plan includes resolution_quality
    if "resolution_quality" not in row["evaluation_plan"]:
        continue

    try:
        response = resolution_chain.invoke(row["transcript"])
        resolution_results.append({
            "call_id": row["call_id"],
            "resolution_quality_score": response.resolution_quality_score,
            "issue_resolved": response.issue_resolved,
            "resolution_justification": response.justification
        })

    except Exception as e:
        print(f"Error evaluating resolution for call {row['call_id']}: {e}")
        resolution_results.append({
            "call_id": row["call_id"],
            "resolution_quality_score": None,
            "issue_resolved": None,
            "resolution_justification": None
        })

resolution_df = pd.DataFrame(resolution_results, columns = ["call_id", "resolution_quality_score", "issue_resolved", "resolution_justification"])
df = df.merge(resolution_df, on = "call_id", how = "left")

print("Resolution Evaluation Applied\n")
print(df[["call_id", "predicted_call_type", "resolution_quality_score", "issue_resolved"]])


Evaluating Resolution: 100%|██████████| 4/4 [00:03<00:00,  1.16it/s]

Resolution Evaluation Applied

   call_id predicted_call_type  resolution_quality_score issue_resolved
0        1             billing                       2.0          False
1        2              claims                       2.0          False
2        3           complaint                       2.0          False
3        4       general_query                       NaN            NaN


In [42]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

min_score, max_score = config["evaluation"]["score_range"]


class KnowledgeEvaluationOutput(BaseModel):
    knowledge_accuracy_score: int = Field(description = f"Knowledge accuracy score between {min_score} and {max_score}", ge = min_score, le = max_score)
    justification: str = Field(description = "Short reason for the score")

knowledge_parser = PydanticOutputParser(pydantic_object = KnowledgeEvaluationOutput)


#prompt template
knowledge_prompt = PromptTemplate(
    template = """
    You are a quality assurance assistant for an insurance customer support team.

    Evaluate the accuracy of the information the agent gave in the following customer support transcript.
    Check whether the agent's answers were correct, clear and relevant to the customer's question,
    and whether the agent avoided guessing or giving misleading information.

    Score it from {min_score} (poor) to {max_score} (excellent).

    Transcript:
    {transcript}

    {format_instructions}
    """,
    input_variables = ["transcript"],
    partial_variables = {
        "format_instructions": knowledge_parser.get_format_instructions(),
            "min_score": min_score,
            "max_score": max_score
                  }
)

#chain

knowledge_chain = knowledge_prompt | llm | knowledge_parser

#test chain with one transcript

knowledge_response = knowledge_chain.invoke(df.iloc[0]["transcript"])

print(f"Knowledge evaluation result: {knowledge_response}")


Knowledge evaluation result: knowledge_accuracy_score=3 justification="The agent acknowledged the customer's issue and indicated they would check it, but did not provide any immediate information or clarification regarding the double charge. The response was relevant but lacked detail and clarity."


In [43]:
from tqdm import tqdm

knowledge_results = []

for i, row in tqdm(df.iterrows(), total = len(df), desc = "Evaluating Knowledge"):
    # only evaluate calls whose evaluation plan includes knowledge_accuracy
    if "knowledge_accuracy" not in row["evaluation_plan"]:
        continue

    try:
        response = knowledge_chain.invoke(row["transcript"])
        knowledge_results.append({
            "call_id": row["call_id"],
            "knowledge_accuracy_score": response.knowledge_accuracy_score,
            "knowledge_justification": response.justification
        })

    except Exception as e:
        print(f"Error evaluating knowledge for call {row['call_id']}: {e}")
        knowledge_results.append({
            "call_id": row["call_id"],
            "knowledge_accuracy_score": None,
            "knowledge_justification": None
        })

knowledge_df = pd.DataFrame(knowledge_results, columns = ["call_id", "knowledge_accuracy_score", "knowledge_justification"])
df = df.merge(knowledge_df, on = "call_id", how = "left")

print("Knowledge Evaluation Applied\n")
print(df[["call_id", "predicted_call_type", "knowledge_accuracy_score"]])


Evaluating Knowledge: 100%|██████████| 4/4 [00:03<00:00,  1.09it/s]

Knowledge Evaluation Applied

   call_id predicted_call_type  knowledge_accuracy_score
0        1             billing                       3.0
1        2              claims                       2.0
2        3           complaint                       NaN
3        4       general_query                       3.0


In [44]:
# from tqdm import tqdm

# def run_evaluations(transcript, eval_plan):
#     results = {}

#     if "tone_empathy" in eval_plan:
#         tone_chain.invoke(transcript})
#         results["tone_empathy_score"] = tone_chain.invoke(transcript).tone_empathy_score
#         results["tone_justification"] = tone_chain.invoke(transcript).justification

#     if "resolution_quality" in eval_plan:
#         resolution_chain.invoke(transcript)
#         results["resolution_quality_score"] = resolution_chain.invoke(transcript).resolution_quality_score
#         results["issue_resolved"] = resolution_chain.invoke(transcript).issue_resolved
#         results["resolution_justification"] = resolution_chain.invoke(transcript).justification

#     if "knowledge_accuracy" in eval_plan:
#         knowledge_chain.invoke(transcript)
#         results["knowledge_accuracy_score"] = knowledge_chain.invoke(transcript).knowledge_accuracy_score
#         results["knowledge_justification"] = knowledge_chain.invoke(transcript).justification

#     return results

In [45]:
from tqdm import tqdm 

def run_evaluations(transcript, eval_plan):
    results = {}

    if "tone_empathy" in eval_plan:
        try:
            tone_result = tone_chain.invoke({"transcript":transcript})
            results["tone"] = tone_result.model_dump()
        except Exception as e:
            results["tone"] = {"error": str(e)}

    if "knowledge_accuracy" in eval_plan:
        try:
            knowledge_result = knowledge_chain.invoke({"transcript":transcript})
            results["knowledge"] = knowledge_result.model_dump()
        except Exception as e:
            results["knowledge"] = {"error": str(e)}

    if "resolution_quality" in eval_plan:
        try:
            resolution_result = resolution_chain.invoke({"transcript":transcript})
            results["resolution"] = resolution_result.model_dump()
        except Exception as e:
            results["resolution"] = {"error": str(e)}


    return results

In [46]:
evaluation_outputs=[]

for i, row in tqdm(df.iterrows(), total = len(df), desc = "Evaluating Calls"):
    output = run_evaluations(row["transcript"], row["evaluation_plan"])

    evaluation_outputs.append({
        "call_id": row["call_id"],
        "evaluation_outputs": output
    })

eval_df = pd.DataFrame(evaluation_outputs)
df = df.merge(eval_df, on = "call_id")

print("evaluation completed\n")

import pprint

pprint.pprint(df.iloc[0]["evaluation_outputs"])


Evaluating Calls: 100%|██████████| 4/4 [00:08<00:00,  2.01s/it]

evaluation completed

{'knowledge': {'justification': "The agent acknowledged the customer's issue "
                                'and indicated they would check it, but did '
                                'not provide any specific information or '
                                'resolution in this part of the transcript. '
                                "Therefore, while the agent's response was "
                                'relevant, it lacked clarity and completeness.',
               'knowledge_accuracy_score': 3},
 'resolution': {'issue_resolved': False,
                'justification': "The agent acknowledged the customer's issue "
                                 'but did not provide a solution or next steps '
                                 'in the provided transcript.',
                'resolution_quality_score': 2}}


In [47]:
class FinalReport(BaseModel):
    summary: str = Field(description = "Summary of the call")
    recommendation: list[str] = Field(description = "List of actionable improvements")

final_parser = PydanticOutputParser(pydantic_object = FinalReport)

final_prompt = PromptTemplate(
    template = """
    You are a QA Manager reviewing customer support calls.

    Based on the evaluation results below, generate:
    1. A concise summary of the agent's performance.
    2. A list of actionable recommendations for improvement. 
    
    Evaluation Data:
    {evaluation_outputs}

    Important: 
    1. Be specific and Practical
    2. Focus on the most important points
    3. Do not repeat scores
    4. Focus on improvement

    {format_instructions}   
    """, 
    input_variables = ["evaluation_outputs"], 
    partial_variables = {
        "format_instructions": final_parser.get_format_instructions(), 
            "labels": config["classification"]["labels"] 
                  }
)

final_chain = final_prompt | llm | final_parser

sample_eval = df.iloc[0]["evaluation_outputs"]

result = final_chain.invoke({"evaluation_outputs": sample_eval})

print("Evaluation Report:\n")
print(result)

Evaluation Report:

summary="The agent demonstrated a basic understanding of the customer's issue but failed to provide specific information or a resolution, resulting in an incomplete support experience." recommendation=['Enhance product knowledge to provide specific solutions or next steps during calls.', 'Practice active listening techniques to ensure all customer concerns are addressed.', 'Utilize a structured approach to problem-solving that includes confirming the issue, suggesting solutions, and outlining next steps.', 'Engage in role-playing scenarios to improve clarity and completeness in responses.']


In [48]:
from tqdm import tqdm

final_results = []

for i, row in tqdm(df.iterrows(), total = len(df), desc = "Generating Final Report"):
    try:
        response = final_chain.invoke({"evaluation_outputs": row["evaluation_outputs"]})
        final_results.append({
            "call_id": row["call_id"],
            "summary": response.summary,
            "recommendation": response.recommendation
        })

    except Exception as e:
        print(f"Error at row {i}: {e}")
        final_results.append({
            "call_id": row["call_id"],
            "summary": None,
            "recommendation": None
        })

final_df = pd.DataFrame(final_results, columns = ["call_id", "summary", "recommendation"])
df = df.merge(final_df, on = "call_id")

print("Final Report Generated\n")


Generating Final Report:   0%|          | 0/4 [00:00<?, ?it/s]

Generating Final Report: 100%|██████████| 4/4 [00:07<00:00,  1.85s/it]

Final Report Generated



In [50]:
print(df[["call_id", "predicted_call_type","evaluation_outputs"]])

   call_id predicted_call_type  \
0        1             billing   
1        2              claims   
2        3           complaint   
3        4       general_query   

                                  evaluation_outputs  
0  {'knowledge': {'knowledge_accuracy_score': 3, ...  
1  {'knowledge': {'knowledge_accuracy_score': 1, ...  
2  {'tone': {'tone_empathy_score': 3, 'justificat...  
3  {'knowledge': {'knowledge_accuracy_score': 3, ...  


In [51]:
accuracy = (df["expected_call_type"] == df["predicted_call_type"]).mean()
print(f"Classification accuracy: {accuracy}")

Classification accuracy: 1.0


In [52]:
df.to_excel("data/output.xlsx", index = False)